# Meshes and time steps: a 1-D experiment

A pulse of some quantity, e.g. a dye or heat, with sharp edges is carried along by a fluid that flows with a constant velocity $U$ from left to right. The exact solution is simple: the pulse moves without changing its shape. This makes it easy to see what the mesh and the time step do to a numerical solution.

The model:

* the domain is 1 m long and is cut into `n_cells` cells of size $\Delta x$; the computer only knows **one value per cell**
* the solver advances the solution in time steps of size `dt` (in OpenFOAM: `deltaT`)
* only every `write_every`-th time step is saved and plotted (in OpenFOAM: `writeInterval`)

You do not need to understand the code in the next cell yet. We will look at how such schemes work later in the course.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

L = 1.0         # length of the domain (m)
U = 1.0         # flow velocity (m/s)
end_time = 0.5  # run time (s)

def pulse(x):
    # Initial condition: a box-shaped pulse between x = 0.1 m and 0.3 m
    return np.where((x > 0.1) & (x < 0.3), 1.0, 0.0)

def run(n_cells, dt, write_every=10, ax=None):
    dx = L / n_cells
    x = (np.arange(n_cells) + 0.5) * dx   # cell centers
    c = pulse(x)                           # one value per cell
    Co = U * dt / dx
    n_steps = int(round(end_time / dt))
    print(f"dx = {dx:.4f} m, dt = {dt:g} s, Courant number Co = {Co:.2f}, {n_steps} time steps")

    if ax is None:
        fig, ax = plt.subplots(figsize=(9, 3.5))
    for step in range(1, n_steps + 1):
        # Upwind scheme: each cell gets some of the content of its upstream neighbor
        c[1:] = c[1:] - Co * (c[1:] - c[:-1])
        c[0] = 0.0
        if step % write_every == 0 or step == n_steps:
            ax.plot(x, c, color="tab:blue", alpha=0.3 + 0.7 * step / n_steps)
    xf = np.linspace(0, L, 1000)
    ax.plot(xf, pulse(xf - U * end_time), "k--", label="exact solution at end time")
    ax.plot(x, c, "o-", color="tab:red", ms=3, label="numerical solution at end time")
    ax.set_xlabel("x (m)")
    ax.set_ylabel("c")
    ax.set_ylim(-0.5, 1.7)
    ax.set_title(f"{n_cells} cells, dt = {dt} s, Co = {Co:.2f}")
    ax.legend(loc="upper left", ncol=2)
    if np.abs(c).max() > 1.7:
        print(f"The solution has blown up: the largest value is {np.abs(c).max():.3g}, it should be 1!")
    return c

## 1. The base case

50 cells and a time step of 0.01 s. The blue lines are the saved time steps, the red line is the result at the end time, and the dashed line is the exact solution.

In [ ]:
run(n_cells=50, dt=0.01);

**Question:** the pulse should keep its shape and its sharp edges. What happens to it instead?

## 2. The mesh

Now change the number of cells. Run the cell below for `n_cells = 25`, `100`, and `400`. With 25 cells, the pulse is only 5 cells wide.

Remember the rule from the lecture: in one time step, the fluid should not move further than one cell, so $Co = U\,\Delta t/\Delta x$ should be smaller than 1. Smaller cells therefore need a smaller time step. Adjust `dt` so that the Courant number stays at 0.5.

**Questions:**

* How does the result change with the number of cells?
* How many time steps does the run need for 400 cells? What does this mean for the run time of a large 3-D model?

In [ ]:
run(n_cells=100, dt=0.005);

## 3. The time step

Keep 100 cells and change only the time step: try `dt = 0.005` ($Co = 0.5$), `0.009`, `0.011`, and `0.02`.

**Questions:**

* What happens when the Courant number is larger than 1?
* What happens with `dt = 0.01` ($Co = 1$ exactly)? This is a special property of this simple 1-D test, do not count on it in a real model!

In [ ]:
run(n_cells=100, dt=0.011);

## 4. Time steps and output

The solver takes many more time steps than we save. Change `write_every` and see how many lines are plotted. The result at the end time does not change, only the number of saved snapshots.

In OpenFOAM, `deltaT` in `system/controlDict` is the time step of the solver, and `writeInterval` controls how often results are written to disk (with `writeControl timeStep;` it counts time steps, just like `write_every` here).

In [ ]:
run(n_cells=100, dt=0.005, write_every=20);

## Summary

* **Mesh:** the computer only knows one value per cell. Features smaller than a few cells are smeared out or lost. More cells give more accurate results, but cost more.
* **Time step:** the time step must fit the mesh. Rule of thumb: $Co = U\,\Delta t/\Delta x < 1$. Smaller cells need smaller time steps, so refining the mesh makes a run more expensive twice.
* **Output:** saving results (`writeInterval`) is independent of the time step of the solver (`deltaT`).

You will see the same effects in OpenFOAM in the cavity flow exercises.